# Load Data

In [1]:
# from myclass_Lung-CLIP import *

from mlxtend.evaluate import bootstrap_point632_score,BootstrapOutOfBag
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
from copy import deepcopy
from numpy import *

import pandas as pd
import numpy as np
import random

#################################################
#------------------ Load Data ------------------#
#################################################

train_scores = pd.read_excel('CLIPScores.validation.xlsx','snvscores_training',engine='openpyxl')
test_scores = pd.read_excel('CLIPScores.validation.xlsx','snvscores_validation',engine='openpyxl')

# clinical_info = pd.read_excel('Clinical Feature.xlsx','Supplementary Table 2',engine='openpyxl').drop(['Subject type', 'Subject group', 'Stage (AJCC v7)', 'Pack-years'], axis=1)
# clinical_info = clinical_info.rename(columns={'Histology':'Histology label'})

In [2]:
train_x = deepcopy(train_scores[['SNVModel_score', 'CNVModel_Score']])
train_x['max_abs'] = [0 for _ in range(len(train_x))]
train_x['max'] = [0 for _ in range(len(train_x))]
for i in range(len(train_x)):
    train_x.loc[i,'max_abs'] = max(abs(train_x.iloc[i]['SNVModel_score']), abs(train_x.iloc[i]['CNVModel_Score']))
    train_x.loc[i,'max'] = max(train_x.iloc[i]['SNVModel_score'], train_x.iloc[i]['CNVModel_Score'])
train_y = train_scores['label']
# train_x.head()

In [3]:
test_x = deepcopy(test_scores[['SNVModel_score', 'CNVModel_Score']])
test_x['max_abs'] = [0 for _ in range(len(test_x))]
test_x['max'] = [0 for _ in range(len(test_x))]
for i in range(len(test_x)):
    test_x.loc[i,'max_abs'] = max(abs(test_x.iloc[i]['SNVModel_score']), abs(test_x.iloc[i]['CNVModel_Score']))
    test_x.loc[i,'max'] = max(test_x.iloc[i]['SNVModel_score'], test_x.iloc[i]['CNVModel_Score'])
test_y = test_scores['label']

test_x.head()

,SNVModel_score,CNVModel_Score,max_abs,max
0,0.164927,0.404171,0.404171,0.404171
1,-0.130548,-0.256987,0.256987,-0.130548
2,-0.039143,1.763853,1.763853,1.763853
3,0.509072,0.367494,0.509072,0.509072
4,0.264437,0.138423,0.264437,0.264437


# lung-CLIP ensemble model

In [4]:
num_inner_iter = 100
num_model = 5
clf_test = np.ones((num_model,len(test_y),num_inner_iter))
# clf_outbag = empty([num_model,num_inner_iter*len(train_y)])
clf_outbag1 = []
clf_outbag2 = []
clf_outbag3 = []
clf_outbag4 = []
clf_outbag5 = []
all_lables_out = []

#-----  Prepare the bootstrap procedure -----#
oob = BootstrapOutOfBag(n_splits=num_inner_iter)
i = 0
for sub_train_index, sub_val_index in oob.split(train_x, train_y):
    
    random.seed(1105+i)
    
    sub_train_x, sub_train_y = train_x.iloc[sub_train_index], train_y.iloc[sub_train_index]
    sub_val_x, sub_val_y = train_x.iloc[sub_val_index], train_y.iloc[sub_val_index]   
    all_lables_out.extend(sub_val_y)
    
    # logistic regression
    clf1 = LogisticRegression().fit(sub_train_x, sub_train_y)
    clf_test[0,:,i] = clf1.predict_proba(test_x)[:,1]
    clf_outbag1.extend(clf1.predict_proba(sub_val_x)[:,1])

    # decision tree
    clf2 = DecisionTreeClassifier().fit(sub_train_x, sub_train_y)
    clf_test[1,:,i] = clf2.predict_proba(test_x)[:,1]
    clf_outbag2.extend(clf2.predict_proba(sub_val_x)[:,1])

    # naive Bayes
    clf3 = GaussianNB().fit(sub_train_x, sub_train_y)
    clf_test[2,:,i] = clf3.predict_proba(test_x)[:,1]
    clf_outbag3.extend(clf3.predict_proba(sub_val_x)[:,1])

    # 5NN
    clf4 = KNeighborsClassifier(n_neighbors=5).fit(sub_train_x, sub_train_y)
    clf_test[3,:,i] = clf4.predict_proba(test_x)[:,1]
    clf_outbag4.extend(clf4.predict_proba(sub_val_x)[:,1])


    # 3NN
    clf5 = KNeighborsClassifier(n_neighbors=3).fit(sub_train_x, sub_train_y)
    clf_test[4,:,i] = clf5.predict_proba(test_x)[:,1]
    clf_outbag5.extend(clf5.predict_proba(sub_val_x)[:,1])
    
    i += 1
    
    
clf_outbag = np.array((clf_outbag1,clf_outbag2,clf_outbag3,clf_outbag4,clf_outbag5))   
sns = empty([num_model])
for i in range(num_model):
    fpr, sen, _= metrics.roc_curve(all_lables_out, clf_outbag[i,:], pos_label=1)
    spe = 1-fpr
    x1 = sen[where(spe>=0.95)[0][0]]
    x2 = sen[where(spe>=0.9)[0][0]]
    x3 = sen[where(spe>=0.85)[0][0]]
    x4 = sen[where(spe>=0.8)[0][0]]
    sns[i] = exp((x1+x2+x3+x4)/4)
sns = np.expand_dims(sns, axis=0)
sns = sns.repeat(len(test_y), axis=0)

std_models = np.ones((len(test_y),num_model))

def mu_std(x):
    mu=mean(x)
    s=std(x)
    return 0 if mu==0 else s/mu
    

for i in range(num_model):
#     std_models[:,i] = clf_test[i,:,:].apply(mu_std,axis=1)
    std_models[:,i] = np.apply_along_axis(mu_std,axis=1,arr=clf_test[i,:,:])
    
def norm(x):
    return x/sum(x)

mysdMdls_rept = np.tile(std_models,num_inner_iter)    
mysdMdls_rept = exp(-mysdMdls_rept*10)*np.tile(sns,num_inner_iter)    
mysdMdls_rept = np.apply_along_axis(norm,axis=1,arr=mysdMdls_rept)

myensemble_pred = clf_test[:,:,0].T
for i in range(1,num_inner_iter):
    myensemble_pred = np.hstack((myensemble_pred,clf_test[:,:,i].T))
    
myscores_full = np.apply_along_axis(sum,axis=1,arr=myensemble_pred*mysdMdls_rept)

In [5]:
metrics.roc_auc_score(test_y,myscores_full)

0.7413949275362318

In [6]:
y_hat = [1 if i>0.5 else 0 for i in myscores_full]
metrics.accuracy_score(test_y, y_hat)

0.6170212765957447